In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Paracetamol"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
print(response.status_code)

200


In [9]:
soup = BeautifulSoup(response.content, "html.parser")
print(soup.prettify()[:2000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Paracetamol - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinne

In [10]:
content = soup.find("div", {"id": "mw-content-text"})


paragraphs = content.find_all("p")


for p in paragraphs[:3]:
    print(p.get_text())
    print("---")



---
Paracetamol,[a] or acetaminophen,[b] is an analgesic and antipyretic agent used to treat fever and mild to moderate pain.[14][15] It is a widely available over-the-counter drug sold generically or under various brand names, including Tylenol and Panadol.

---
Paracetamol relieves pain in both acute mild migraine and episodic tension headache.[16][17] The aspirin/paracetamol/caffeine combination also helps with both conditions when the pain is mild and is recommended as a first-line treatment for them.[18] At a standard dose, paracetamol slightly reduces fever,[14][19][20] though it is inferior to ibuprofen in that respect[21] and the benefits of its use for fever are unclear, particularly in the context of fever of viral origins.[22] Paracetamol is effective for pain after wisdom tooth extraction, but it is less effective than ibuprofen.[23] The combination of paracetamol and ibuprofen provides greater analgesic efficacy than either drug alone.[23][24] The pain relief paracetamol

In [20]:
import re

def clean_text(text):
    text = re.sub(r'\[[\d+\w]+\]', '', text)  # remove citations like [14]
    text = re.sub(r'\s+', ' ', text) # remove extra whitespace
    return text.strip()

for p in paragraphs[:3]:
    print(clean_text(p.get_text()))
    print("---")


---
Paracetamol, or acetaminophen, is an analgesic and antipyretic agent used to treat fever and mild to moderate pain. It is a widely available over-the-counter drug sold generically or under various brand names, including Tylenol and Panadol.
---
Paracetamol relieves pain in both acute mild migraine and episodic tension headache. The aspirin/paracetamol/caffeine combination also helps with both conditions when the pain is mild and is recommended as a first-line treatment for them. At a standard dose, paracetamol slightly reduces fever, though it is inferior to ibuprofen in that respect and the benefits of its use for fever are unclear, particularly in the context of fever of viral origins. Paracetamol is effective for pain after wisdom tooth extraction, but it is less effective than ibuprofen. The combination of paracetamol and ibuprofen provides greater analgesic efficacy than either drug alone. The pain relief paracetamol provides in osteoarthritis is small and clinically insignif

# was able to manual scrap data from wikipedia regarding medicines

In [16]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

driver.get("https://www.1mg.com/drugs/crocin-advance-500mg-tablet-152074")
time.sleep(5)

print(driver.title)
print(driver.current_url)

Buy Entozole Eye Drop Online: View Uses, Side Effects, Price, Substitutes | 1mg
https://www.1mg.com/drugs/entozole-eye-drop-152074


In [15]:
import time

driver.get("https://www.1mg.com/drugs/crocin-advance-500mg-tablet-152074")
time.sleep(3)

print(driver.title)
print(driver.current_url)

Buy Entozole Eye Drop Online: View Uses, Side Effects, Price, Substitutes | 1mg
https://www.1mg.com/drugs/entozole-eye-drop-152074


In [17]:
driver.quit()

In [18]:
# NOTE: 1mg bot detection bypass attempted
# 1mg detects Selenium via "Chrome is being controlled by automated test software" banner
# Redirects to random medicine pages instead of requested URL
# Solution: Use OpenFDA API instead (more reliable, medically accurate)
# TODO: Revisit 1mg scraping with rotating proxies + mouse simulation

In [12]:
import requests

url = "https://api.fda.gov/drug/label.json?search=openfda.generic_name:acetaminophen&limit=1"
response = requests.get(url)
data = response.json()

print(data.keys())

dict_keys(['meta', 'results'])


In [14]:
print(type(data['results']))
print(len(data['results']))
print(type(data['results'][0]))

<class 'list'>
1
<class 'dict'>


In [15]:
medicine = data['results'][0]
print(medicine.keys())

dict_keys(['spl_product_data_elements', 'active_ingredient', 'purpose', 'indications_and_usage', 'warnings', 'do_not_use', 'ask_doctor', 'ask_doctor_or_pharmacist', 'stop_use', 'pregnancy_or_breast_feeding', 'keep_out_of_reach_of_children', 'dosage_and_administration', 'storage_and_handling', 'inactive_ingredient', 'questions', 'package_label_principal_display_panel', 'set_id', 'id', 'effective_time', 'version', 'openfda'])


In [18]:
print(type(medicine['indications_and_usage']))
print(medicine['indications_and_usage'])

<class 'list'>
['Uses temporarily relieves minor aches and pains due to: headache the common cold backache minor pain of arthritis toothache muscular aches premenstrual and menstrual cramps temporarily reduces fever']


In [20]:
fields = {
    "name": medicine['openfda']['generic_name'][0],
    "purpose": medicine.get('purpose', ['N/A'])[0],
    "uses": medicine.get('indications_and_usage', ['N/A'])[0],
    "dosage": medicine.get('dosage_and_administration', ['N/A'])[0],
    "warnings": medicine.get('warnings', ['N/A'])[0],
    "do_not_use": medicine.get('do_not_use', ['N/A'])[0],      
    "stop_use": medicine.get('stop_use', ['N/A'])[0],          
    "ask_doctor": medicine.get('ask_doctor', ['N/A'])[0],     
    "pregnancy": medicine.get('pregnancy_or_breast_feeding', ['N/A'])[0],
    "active_ingredient": medicine.get('active_ingredient', ['N/A'])[0],
    "questions": medicine.get('questions', ['N/A'])[0]
}

for key, value in fields.items():
    print(f"{key.upper()}:")
    print(value[:200])
    print("---")

NAME:
ACETAMINOPHEN
---
PURPOSE:
Purpose Pain reliever/fever reducer
---
USES:
Uses temporarily relieves minor aches and pains due to: headache the common cold backache minor pain of arthritis toothache muscular aches premenstrual and menstrual cramps temporarily reduces fever
---
DOSAGE:
Directions do not take more than directed adults and children 12 years and over take 2 gelcaps every 6 hours while symptoms last do not take more than 6 gelcaps in 24 hours, unless directed by a docto
---
WARNINGS:
Warnings Liver warning: This product contains acetaminophen. Severe liver damage may occur if you take more than 4,000 mg of acetaminophen in 24 hours with other drugs containing acetaminophen 3 or mo
---
DO_NOT_USE:
Do not use if you are allergic to acetaminophen or any of the inactive ingredients in this product with any other drug containing acetaminophen (prescription or nonprescription). If you are not sure w
---
STOP_USE:
Stop use and ask a doctor if new symptoms occur pain gets worse

In [21]:
def create_chunks(fields):
    name = fields['name']
    chunks = []
    
    for key, value in fields.items():
        if key != 'name' and value != 'N/A':
            chunk = f"{name} {key}: {value}"
            chunks.append(chunk)
    
    return chunks

chunks = create_chunks(fields)

for chunk in chunks:
    print(chunk[:150])
    print("---")

ACETAMINOPHEN purpose: Purpose Pain reliever/fever reducer
---
ACETAMINOPHEN uses: Uses temporarily relieves minor aches and pains due to: headache the common cold backache minor pain of arthritis toothache muscul
---
ACETAMINOPHEN dosage: Directions do not take more than directed adults and children 12 years and over take 2 gelcaps every 6 hours while symptoms last
---
ACETAMINOPHEN warnings: Warnings Liver warning: This product contains acetaminophen. Severe liver damage may occur if you take more than 4,000 mg of a
---
ACETAMINOPHEN do_not_use: Do not use if you are allergic to acetaminophen or any of the inactive ingredients in this product with any other drug conta
---
ACETAMINOPHEN stop_use: Stop use and ask a doctor if new symptoms occur pain gets worse or lasts more than 10 days fever gets worse or lasts more than
---
ACETAMINOPHEN ask_doctor: Ask a doctor before use if you have liver disease.
---
ACETAMINOPHEN pregnancy: If pregnant or breast-feeding, ask a health professional

In [ ]:
cu